In [1]:
from farmdar.secrets import set_aws_keys
from farmdar.auth import refresh_token
import s3fs
from farmdar.data import insert_df
import dask_geopandas as dgpd
import dask
from dask.distributed import Client
import re
import geopandas as gpd
import pandas as pd
from datetime import datetime
set_aws_keys()

In [2]:
import rasterio
from farmdar.secrets import set_aws_keys

set_aws_keys()

# GDAL-compatible virtual path
vsipath = "/vsis3/centralized-data-storage/BAT/b_skywatch_raw/1_2023_2023-04-29_BAT/1_1_3m_2023_2023-04-29_BAT_20230429T0503_573f.tif"

with rasterio.Env():  # No manual credentials here!
    with rasterio.open(vsipath) as src:
        data = src.read()
        profile = src.profile


In [3]:
import numpy as np
import xarray as xr

# Assuming `data` shape: (bands, height, width)
bands, height, width = data.shape
transform = profile['transform']

# Coordinates
x_coords = np.arange(width) * transform[0] + transform[2]
y_coords = np.arange(height) * transform[4] + transform[5]
band_coords = list(range(1, bands + 1))  # e.g. 1 to 8

# Build DataArray and convert to Dataset
da = xr.DataArray(
    data,
    coords={"band": band_coords, "y": y_coords, "x": x_coords},
    dims=["band", "y", "x"],
    name="values"
)

ds = da.to_dataset(name="values")

# Save to S3 as Zarr
zarr_s3_path = "s3://centralized-data-storage/zarr/1_1_3m_2023_2023-04-29_BAT_20230429T0503_573f"
ds.to_zarr(zarr_s3_path, mode='w')  # ✅ full dataset with 'values'


In [4]:
import xarray as xr
import fsspec

fs_map = fsspec.get_mapper(zarr_s3_path, anon=False)

zds = xr.open_zarr(fs_map, consolidated=False)
print(zds)                     # should list 'values'
zarr_data = zds["values"]      # ✅ this now works
print(zarr_data.shape)         # should be (8, H, W)


<xarray.Dataset> Size: 92MB
Dimensions:  (band: 8, y: 1232, x: 2334)
Coordinates:
  * band     (band) int64 64B 1 2 3 4 5 6 7 8
  * x        (x) float64 19kB 72.47 72.47 72.47 72.47 ... 72.54 72.54 72.54
  * y        (y) float64 10kB 34.49 34.49 34.49 34.49 ... 34.46 34.45 34.45
Data variables:
    values   (band, y, x) float32 92MB dask.array<chunksize=(2, 308, 584), meta=np.ndarray>
(8, 1232, 2334)
